In [33]:
# PV Pricing using Fuzzy Inference System (Sugeno)
# Author: Fakhri Apriansyah

In [34]:
import numpy as np

def trimf(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    elif x == b:
        return 1.0
    elif x < b:
        return (x - a) / (b - a + 1e-6)
    else:
        return (c - x) / (c - b + 1e-6)


def trapmf(x, a, b, c, d):
    if x <= a or x >= d:
        return 0.0
    elif b <= x <= c:
        return 1.0
    elif a < x < b:
        return (x - a) / (b - a + 1e-6)
    else:
        return (d - x) / (d - c + 1e-6)


def fuzzify_area(x):
    small = trapmf(x, 0, 0, 20, 40)
    medium = trimf(x, 20, 50, 80)
    large = trapmf(x, 60, 80, 100, 100)
    return small, medium, large


def fuzzify_power(x):
    low = trapmf(x, 0, 0, 5, 10)
    medium = trimf(x, 5, 15, 25)
    high = trapmf(x, 20, 30, 40, 40)
    return low, medium, high

In [35]:
def evaluate_fis(area, power):
    a_small, a_med, a_large = fuzzify_area(area)
    p_low, p_med, p_high = fuzzify_power(power)

    rules = [
        (min(a_small, p_low), BASIC),
        (min(a_small, p_med), BASIC),
        (min(a_small, p_high), PREMIUM),

        (min(a_med, p_low), BASIC),
        (min(a_med, p_med), PREMIUM),
        (min(a_med, p_high), PLATINUM),

        (min(a_large, p_low), PREMIUM),
        (min(a_large, p_med), PLATINUM),
        (min(a_large, p_high), PLATINUM),
    ]

    num = sum(w*z for w, z in rules)
    den = sum(w for w, z in rules)

    if den == 0:
        return 1  # BASIC fallback

    return num / den

In [36]:
# ==============================
# Single Test Case
# ==============================

area = 30
power = 18

result = evaluate_fis(area, power)

print("=== SINGLE TEST CASE ===")
print(f"Input Area   : {area} m²")
print(f"Input Power  : {power} kW")
print(f"FIS Output   : {result:.2f}")

# Classification
if result < 1.5:
    label = "BASIC"
elif result < 2.5:
    label = "PREMIUM"
else:
    label = "PLATINUM"

print(f"Package      : {label}")

=== SINGLE TEST CASE ===
Input Area   : 30 m²
Input Power  : 18 kW
FIS Output   : 1.40
Package      : BASIC
